# Phase 1: BTC Data Download, Cleaning, Indicators, Visualization

This notebook builds the first dataset for the RL environment.

**Goals**:
- Download BTC-USD daily OHLCV data
- Clean and validate data
- Add RSI, MACD, and moving averages
- Visualize price and indicators

## Step 0: Install dependencies
If you have not installed the dependencies, run:

```bash
pip install -r requirements.txt
```

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from ta.momentum import RSIIndicator
from ta.trend import MACD, SMAIndicator

## Step 1: Download BTC daily data
We use yfinance for a beginner-friendly data source.
The data is for learning and prototyping, not for live trading.

In [ ]:
symbol = "BTC-USD"
start_date = "2018-01-01"
end_date = "2024-12-31"
interval = "1d"

df = yf.download(symbol, start=start_date, end=end_date, interval=interval, auto_adjust=False)
df = df.rename(columns=str.title)
df.head()

## Step 2: Clean and validate
We remove missing rows and ensure expected columns exist.

In [ ]:
required_cols = ["Open", "High", "Low", "Close", "Volume"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df.dropna().copy()
df.describe()

## Step 3: Add indicators
We add RSI, MACD, and two simple moving averages.

In [ ]:
df["RSI_14"] = RSIIndicator(close=df["Close"], window=14).rsi()
macd = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
df["MACD"] = macd.macd()
df["MACD_Signal"] = macd.macd_signal()
df["MA_10"] = SMAIndicator(close=df["Close"], window=10).sma_indicator()
df["MA_50"] = SMAIndicator(close=df["Close"], window=50).sma_indicator()

df = df.dropna().copy()
df.tail()

## Step 4: Visualize price and indicators
We plot price and indicators to confirm they look reasonable.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df.index, df["Close"], label="Close", linewidth=1)
plt.plot(df.index, df["MA_10"], label="MA 10", linewidth=1)
plt.plot(df.index, df["MA_50"], label="MA 50", linewidth=1)
plt.title("BTC-USD Close with Moving Averages")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df.index, df["RSI_14"], label="RSI 14")
plt.axhline(70, color="red", linestyle="--", linewidth=1)
plt.axhline(30, color="green", linestyle="--", linewidth=1)
plt.title("RSI 14")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df.index, df["MACD"], label="MACD")
plt.plot(df.index, df["MACD_Signal"], label="Signal")
plt.title("MACD")
plt.legend()
plt.tight_layout()
plt.show()

## Step 5: Save the processed dataset
We store a clean CSV for later phases.

In [ ]:
output_dir = Path("..") / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "btc_1d_indicators.csv"
df.to_csv(output_path)
output_path